# 美股第二階段：時機評估與 AI 研究

獨立使用 Yahoo、美元與紐約交易時間。接收 WHID 美股候選，新增市場／個股／新聞情緒與TimesFM 10日價格路徑研究，不修改基本面／估值分數、既有交易建議或AI特徵，也不自動下單。量價參考不是機構實際買賣超。請先讀 README_美股第二階段.md。

In [ ]:
from pathlib import Path
import sys, importlib.util

# 可攜式定位：可從專案根目錄或 Stock_price_prediction 目錄開啟。
# 如果另存到其他專案，會使用該專案的相對位置，不綁定電腦上的絕對路徑。
START = Path.cwd().resolve()
search = [START, START / "Stock_price_prediction"]
for parent in START.parents:
    search.extend([parent, parent / "Stock_price_prediction"])
BASE = next((p for p in search if (p / "timing_us.py").is_file() and (p / "config").is_dir()), None)
if BASE is None:
    raise FileNotFoundError("找不到 Stock_price_prediction 程式目錄；請從專案根目錄或該資料夾開啟 Notebook")
BASE = BASE.resolve()
PROJECT_ROOT = BASE.parent

# 更新後即使未重啟Kernel，也不沿用先前從其他資料夾載入的同名模組。
module_prefixes = ("timing_us", "timing_timesfm") if "US" == "US" else ("timing_data", "timing_rules", "timing_ai", "timing_tw", "timing_timesfm")
for module_name in list(sys.modules):
    if any(module_name == p or module_name.startswith(p + ".") or module_name.startswith(p + "_") for p in module_prefixes):
        del sys.modules[module_name]
sys.path[:] = [p for p in sys.path if not (p and Path(p).name == "Stock_price_prediction")]
sys.path.insert(0, str(BASE.resolve()))

needed = {"numpy":"numpy", "pandas":"pandas", "yaml":"PyYAML", "yfinance":"yfinance",
          "openpyxl":"openpyxl", "sklearn":"scikit-learn", "xgboost":"xgboost", "torch":"torch", "timesfm3":"timesfm[torch]"}
missing = [package for module, package in needed.items() if importlib.util.find_spec(module) is None]
print("缺少套件：", missing or "無")
print("本次專案：", PROJECT_ROOT.resolve())
print("程式目錄：", BASE.resolve())
print("套件清單：", BASE / "requirements-timing-US.txt")


In [ ]:
# 如需安裝才改為 True，完成後重啟 Kernel。
INSTALL_MISSING = False
if INSTALL_MISSING:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(BASE / "requirements-timing-US.txt")])
    print("安裝完成，請重啟Kernel並重新執行。")


In [ ]:
import pandas as pd
import timing_us_data, timing_us, timing_timesfm
from timing_us_data import VERSION, load_config, candidates
from timing_us import run, finlab_research_positions
for module in [timing_us_data, timing_us, timing_timesfm]:
    if Path(module.__file__).resolve().parent != BASE.resolve():
        raise RuntimeError(f"模組路徑不一致：{module.__file__}")
print("程式版本：", VERSION)
print("資料模組：", Path(timing_us_data.__file__).resolve())
CONFIG_PATH = BASE / "config" / "timing_US.yaml"
TICKERS = []  # 例如 ["AAPL", "NVDA"]；空白則讀取最新WHID美股報表。
settings = load_config(CONFIG_PATH)
if TICKERS:
    settings["candidates"]["tickers"] = TICKERS
candidate_frame, source_info = candidates(settings, PROJECT_ROOT)
print(source_info)
display(candidate_frame)


## 分析與分開輸出

目前依 YAML 處理全部候選；可用 max_stocks 限制檔數。下載及訓練需要時間。最新日缺行情時，會在期限內顯示有日期的最近完整日歷史參考，並註明等待最新行情確認；不會補猜收盤價。AI三分類樣本不足時保留空值，不影響技術規則判讀。確認版本為 US-timing-1.8。

In [ ]:
result = run(CONFIG_PATH, candidate_frame=candidate_frame, provenance=source_info)
display(result["simple"])
for kind, path in result["paths"].items():
    print(kind, path)


In [ ]:
# 固定候選名單的歷史研究，不是WHID歷史選股或多檔組合績效。
display(result["backtest"])
display(result["ai_status"])
display(result["data_status"])


## 可選：FinLab 美股研究介接（預設關閉）

需自行安裝FinLab及登入有美股資料權限的帳號。不同引擎的成交、持有期、權重與費用假設需要另行核對。這段未使用付費帳號實跑，不會下單、上傳報告或發送通知。

In [ ]:
RUN_FINLAB = False
if RUN_FINLAB:
    from finlab.backtest import sim
    # 登入依官方流程在本機進行，不要將金鑰寫進 Notebook。
    positions = finlab_research_positions(result["signals"])
    if positions.empty or result["backtest"].empty:
        raise ValueError("沒有可用研究訊號")
    start = pd.to_datetime(result["backtest"]["start"]).max()
    end = pd.to_datetime(result["backtest"]["end"]).min()
    positions = positions.loc[start:end]
    if positions.empty:
        raise ValueError("各檔有效研究期間沒有交集")
    finlab_report = sim(positions, market="US_STOCK", trade_at_price="open", position_limit=0.1,
        fee_ratio=settings["backtest"]["fee_rate"], tax_ratio=settings["backtest"]["sell_fee_rate"],
        stop_loss=settings["backtest"]["stop_loss"], take_profit=settings["backtest"]["take_profit"],
        upload=False, notification_enable=False, name="WHID美股固定名單時機研究")
    display(finlab_report)
